# Large Volume & Block Distribution — Interactive Lab

The previous labs used tiny files (Titanic ~61 KB). A single small file fits in **one HDFS block on one DataNode** — so you never actually *see* the thing Hadoop was built for: **splitting big data into blocks and spreading replicas across the cluster**.

Here we ingest a **multi‑GB** public dataset (NYC Yellow Taxi trips, Parquet) and then make the distributed storage **visible**: blocks, replicas, where they live, and what happens when a DataNode dies.

**Dataset:** [NYC TLC Trip Records](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) — public, no authentication, ~50 MB per month.

> Some steps (`fsck`, `dfsadmin`, stopping a DataNode) are **not** available through the WebHDFS proxy. They run inside the cluster containers via the `hadoop()` helper below — the same idea as `make shell-namenode`, just orchestrated from Python.

## Setup

Connect to HDFS via the proxy, and define a helper to run admin/cluster commands inside the containers.

In [ ]:
import sys; sys.path.insert(0, '../scripts')
from hdfs_utils import hadoop, compose
from hdfs import InsecureClient

client = InsecureClient('http://localhost:14000', user='root')
print('Connected to HDFS via proxy ✓')


## 1. Configure the ingestion volume

Each monthly Parquet file is ~50 MB. **Start small** (1–2 months) to validate the flow, then raise `MONTHS` to reach ~1–3 GB once everything works.

| Months | Approx. size on HDFS (×3 replication) |
|---|---|
| 2 | ~100 MB raw → ~300 MB stored |
| 12 | ~600 MB raw → ~1.8 GB stored |
| 24 | ~1.2 GB raw → ~3.6 GB stored |

In [ ]:
# Build the list of months to download. Yellow taxi data is published with a
# ~2 month lag; 2023 is a safe, fully-available year.
YEAR = 2023
N_MONTHS = 2          # ↑ raise to 12–24 for a true 1–3 GB demo once the flow works

MONTHS = [f'{YEAR}-{m:02d}' for m in range(1, N_MONTHS + 1)]
BASE_URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{}.parquet'
HDFS_DIR = '/datasets/nyc_taxi'

print(f'Will ingest {len(MONTHS)} month(s): {MONTHS}')

## 2. Download in streaming (chunked) mode

Large files must **not** be loaded fully into memory. We stream them to `temp/` in 1 MB chunks. `temp/` is git-ignored.

In [ ]:
from hdfs_utils import download_stream
os.makedirs('../temp', exist_ok=True)


## 3. Ingest into HDFS

Upload every month into one HDFS directory. With the default 128 MB block size and replication factor 3, the NameNode fragments each file into blocks and scatters **3 replicas of every block** across the 3 DataNodes.

### CLI equivalent
```bash
hdfs dfs -mkdir -p /datasets/nyc_taxi
hdfs dfs -put ../temp/yellow_*.parquet /datasets/nyc_taxi/
```

In [ ]:
client.makedirs(HDFS_DIR, permission=0o755)

for local in local_files:
    name = os.path.basename(local)
    client.upload(f'{HDFS_DIR}/{name}', local, overwrite=True)
    print(f'uploaded → {HDFS_DIR}/{name}')

# Total size stored in HDFS (logical size, before ×3 replication)
listing = client.list(HDFS_DIR, status=True)
logical = sum(s['length'] for _, s in listing)
print(f'\n{len(listing)} files in {HDFS_DIR} — {logical / 1e6:.1f} MB logical')
print(f'≈ {logical * 3 / 1e6:.1f} MB physical across the cluster (replication ×3)')

## 4. See the fragmentation: blocks and their locations

`hdfs fsck` is the canonical way to inspect how a path is split into blocks and which DataNodes hold each replica.

### CLI equivalent
```bash
hdfs fsck /datasets/nyc_taxi -files -blocks -locations
```

In [ ]:
report = hadoop(f'hdfs fsck {HDFS_DIR} -files -blocks -locations')
print(report[-4000:])  # tail: summary with Total blocks / replication / health

Look for **`Total blocks`**, **`Average block replication`** (should be ~3), and **`Corrupt blocks: 0`**. Each block line lists the DataNode IPs holding its replicas — proof the data is spread across the cluster.

👉 You can also browse this visually in the NameNode UI: **[localhost:9870 → Utilities → Browse the file system](http://localhost:9870/explorer.html#/datasets/nyc_taxi)** (click a file → *Block information*).

## 5. Replication & cluster health (the DataNode report)

### CLI equivalent
```bash
hdfs dfsadmin -report
```

In [ ]:
report = hadoop('hdfs dfsadmin -report')
# Show the cluster summary + per-DataNode usage
print(report[:3000])

Notice **`Live datanodes (3)`** and that each node reports roughly the **same DFS Used** — HDFS balances replicas so the load is shared, not piled on one machine.

## 6. (Didactic) Force a single file to split into many blocks

A 50 MB file is smaller than the 128 MB default block, so it's just **one** block. To *see* one file fragment into several blocks, upload a copy with a small block size (16 MB). WebHDFS accepts a per-write `blocksize` (must be a multiple of 512).

In [ ]:
small_block = 16 * 1024 * 1024  # 16 MB
demo_src = local_files[0]
demo_dst = f'{HDFS_DIR}/_blocksize_demo.parquet'

client.upload(demo_dst, demo_src, overwrite=True, blocksize=small_block)
print(f'Uploaded {os.path.basename(demo_src)} with a 16 MB block size.')

report = hadoop(f'hdfs fsck {demo_dst} -files -blocks')
print(report[-1500:])

A ~50 MB file now shows **4 blocks** instead of 1 — the same fragmentation that, at petabyte scale, lets thousands of machines read one file in parallel.

## 7. Fault tolerance: kill a DataNode, watch HDFS heal

Replication exists so the cluster survives hardware failure. We stop `datanode3`, then check the file system — blocks that lived only on that node become **under-replicated**, and HDFS automatically re-replicates them onto the survivors.

> This uses `docker compose` from the project root, so the helper targets the host, not a container.

In [ ]:
# compose() imported from hdfs_utils


In [ ]:
import time

# Give the NameNode a few seconds to notice and start healing.
time.sleep(10)
report = hadoop(f'hdfs fsck {HDFS_DIR}')
print(report[-2000:])
print('\nLook for "Under-replicated blocks" > 0 — HDFS is rebuilding the missing replicas.')

In [ ]:
# Bring the node back — the cluster returns to full health on its own.
print(compose('start datanode3'))
print('\nRestarted datanode3. Re-run the fsck cell above after ~30s: under-replicated → 0.')

## 8. Read it back: analytics straight from the Data Lake

The data never left HDFS. We stream one month back through the proxy into a Pandas DataFrame (Parquet needs `pyarrow`).

In [ ]:
import io
import pandas as pd

first = client.list(HDFS_DIR)[0]
with client.read(f'{HDFS_DIR}/{first}') as reader:
    df = pd.read_parquet(io.BytesIO(reader.read()))

print(f'{first}: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# A quick aggregation: average tip by passenger count
cols = [c for c in ['passenger_count', 'tip_amount'] if c in df.columns]
if len(cols) == 2:
    summary = (df.groupby('passenger_count')['tip_amount']
                 .mean().round(2).head(10))
    print('Average tip by passenger count:')
    print(summary)
else:
    print('Columns vary by year; available:', list(df.columns))

## 9. Cleanup

Remove the dataset from HDFS and the local downloads. Skip this if you want to keep the data for labs 05/06.

In [ ]:
from hdfs_utils import safe_delete

safe_delete(client, '/datasets/nyc_taxi')
for f in local_files:
    if os.path.exists(f):
        os.remove(f)
print('Removed local temp parquet files ✓')


## Summary

| Concept | How we observed it |
|---|---|
| **Fragmentation into blocks** | `hdfs fsck -files -blocks` + the 16 MB block-size demo |
| **Replication (×3)** | `hdfs dfsadmin -report` + `Average block replication` in fsck |
| **Distribution across nodes** | per-block DataNode locations; balanced `DFS Used` per node |
| **Fault tolerance** | stopped `datanode3` → under-replicated → auto-heal |
| **Read-back analytics** | `pd.read_parquet` streamed from HDFS via the proxy |